Script permettant de télécharger openmétéo

In [4]:
"""Téléchargement des données météo horaires (Open-Meteo, API Archive) pour
12 grandes villes françaises (une par région), de 2021 à 2025.

Usage :
    python telecharger_meteo.py

Résultat :
    data/raw/openmeteo/<ville>_<année>.json   (réponses brutes, servent de cache)
    data/processed/meteo_horaire.parquet      (tableau final, une ligne par ville et par heure)

Dépendances : requests, pandas, pyarrow
Données : Open-Meteo.com (licence CC BY 4.0, usage non commercial) à citer dans le rapport.
"""

import json
import time
from pathlib import Path

import pandas as pd
import requests

URL = "https://archive-api.open-meteo.com/v1/archive"
RAW_DIR = Path("data/raw/openmeteo")
OUT_PATH = Path("data/processed/meteo_horaire.parquet")

ANNEES = range(2021, 2026)  # 2021 à 2025 inclus

HOURLY = [
    "temperature_2m",
    "shortwave_radiation",
    "cloud_cover",
    "wind_speed_10m",
    "wind_speed_100m",
    "precipitation",
]

# Une ville représentative par région (France métropolitaine hors Corse).
VILLES = {
    "paris":      {"lat": 48.8566, "lon": 2.3522,  "region": "Île-de-France"},
    "lyon":       {"lat": 45.7640, "lon": 4.8357,  "region": "Auvergne-Rhône-Alpes"},
    "toulouse":   {"lat": 43.6047, "lon": 1.4442,  "region": "Occitanie"},
    "bordeaux":   {"lat": 44.8378, "lon": -0.5792, "region": "Nouvelle-Aquitaine"},
    "lille":      {"lat": 50.6292, "lon": 3.0573,  "region": "Hauts-de-France"},
    "strasbourg": {"lat": 48.5734, "lon": 7.7521,  "region": "Grand Est"},
    "marseille":  {"lat": 43.2965, "lon": 5.3698,  "region": "Provence-Alpes-Côte d'Azur"},
    "nantes":     {"lat": 47.2184, "lon": -1.5536, "region": "Pays de la Loire"},
    "rennes":     {"lat": 48.1173, "lon": -1.6778, "region": "Bretagne"},
    "rouen":      {"lat": 49.4432, "lon": 1.0993,  "region": "Normandie"},
    "dijon":      {"lat": 47.3220, "lon": 5.0415,  "region": "Bourgogne-Franche-Comté"},
    "orleans":    {"lat": 47.9029, "lon": 1.9093,  "region": "Centre-Val de Loire"},
}


def fetch_year(nom, lat, lon, annee, max_retries=5):
    """Télécharge une année pour une ville. Utilise le cache disque s'il existe."""
    path = RAW_DIR / f"{nom}_{annee}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": f"{annee}-01-01",
        "end_date": f"{annee}-12-31",
        "hourly": ",".join(HOURLY),
        "timezone": "UTC",  # évite les problèmes de changement d'heure
    }

    for tentative in range(max_retries):
        attente = 10 * 2**tentative
        try:
            r = requests.get(URL, params=params, timeout=60)
        except requests.exceptions.RequestException as e:
            # coupure réseau, erreur SSL, timeout... : on attend puis on réessaie
            print(f"  Erreur réseau ({type(e).__name__}), nouvelle tentative dans {attente}s")
            time.sleep(attente)
            continue

        if r.status_code == 200:
            RAW_DIR.mkdir(parents=True, exist_ok=True)
            path.write_text(r.text, encoding="utf-8")
            return r.json()
        if r.status_code == 429 or r.status_code >= 500:
            # limite d'appels atteinte ou erreur serveur : on attend puis on réessaie
            print(f"  Code {r.status_code}, nouvelle tentative dans {attente}s")
            time.sleep(attente)
            continue
        r.raise_for_status()

    raise RuntimeError(f"Échec du téléchargement pour {nom} {annee}")


def to_dataframe(nom, info, payload):
    df = pd.DataFrame(payload["hourly"])
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df["ville"] = nom
    df["region"] = info["region"]
    return df


def main(pause=1.0):
    frames = []
    for nom, info in VILLES.items():
        for annee in ANNEES:
            print(f"{nom} {annee}")
            payload = fetch_year(nom, info["lat"], info["lon"], annee)
            frames.append(to_dataframe(nom, info, payload))
            time.sleep(pause)  # politesse envers l'API gratuite

    meteo = pd.concat(frames, ignore_index=True)

    # Contrôle : 5 ans d'heures par ville (43 824 lignes), sans doublons
    attendu = len(pd.date_range("2021-01-01", "2025-12-31 23:00", freq="h"))
    comptes = meteo.groupby("ville").size()
    anormales = comptes[comptes != attendu]
    if not anormales.empty:
        print(f"Attention, nombre de lignes inattendu (attendu {attendu}) :\n{anormales}")
    assert not meteo.duplicated(["ville", "time"]).any(), "Doublons détectés"

    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    meteo.to_parquet(OUT_PATH, index=False)
    print(f"{len(meteo)} lignes enregistrées dans {OUT_PATH}")


if __name__ == "__main__":
    main()

paris 2021


paris 2022
paris 2023
paris 2024
paris 2025
lyon 2021
lyon 2022
lyon 2023
lyon 2024
lyon 2025
toulouse 2021
toulouse 2022
toulouse 2023
toulouse 2024
toulouse 2025
bordeaux 2021
bordeaux 2022
bordeaux 2023
bordeaux 2024
bordeaux 2025
lille 2021
lille 2022
lille 2023
lille 2024
lille 2025
strasbourg 2021
strasbourg 2022
strasbourg 2023
strasbourg 2024
strasbourg 2025
marseille 2021
marseille 2022
marseille 2023
marseille 2024
marseille 2025
nantes 2021
nantes 2022
nantes 2023
nantes 2024
nantes 2025
rennes 2021
rennes 2022
rennes 2023
rennes 2024
rennes 2025
  Code 429, nouvelle tentative dans 10s
rouen 2021
rouen 2022
rouen 2023
rouen 2024
rouen 2025
dijon 2021
dijon 2022
dijon 2023
  Erreur réseau (SSLError), nouvelle tentative dans 10s
dijon 2024
  Erreur réseau (ConnectTimeout), nouvelle tentative dans 10s
dijon 2025
orleans 2021
orleans 2022
orleans 2023
orleans 2024
orleans 2025
525888 lignes enregistrées dans data/processed/meteo_horaire.parquet


,temperature_2m,shortwave_radiation,cloud_cover,wind_speed_10m,wind_speed_100m,precipitation
count,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000,8760.000000
mean,11.448744,137.443037,62.826484,12.391016,21.006701,0.086119
std,6.906839,207.937454,38.883546,6.552435,10.046572,0.332811
min,-6.000000,0.000000,0.000000,0.400000,0.400000,0.000000
25%,6.300000,0.000000,25.000000,7.600000,13.700000,0.000000
50%,11.000000,7.000000,79.000000,10.900000,20.400000,0.000000
75%,16.600000,222.000000,100.000000,16.200000,27.000000,0.000000
max,32.100000,905.000000,100.000000,52.400000,76.700000,7.900000
